# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object allows dot access.
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

> **Note**: All entities are referenced by their `@id` as per FAIR and Croissant best practices.

In [ ]:
# Print all available record sets and their @id
print("Available record sets in dataset (by @id):")
for rset in metadata.record_sets:
    print(f"  Record set: {rset['@id']}")
    print(f"    Name: {rset['name']}\n    Description: {rset.get('description', 'No description')}\n    Columns:")
    fields = rset.get('fields', [])
    if not fields:  # If 'fields' is missing, try 'columns'
        fields = rset.get('columns', [])
    for f in fields:
        # 'f' is a dict like: {'@id': 'someid'}, resolve field details by @id
        field_obj = next((fld for fld in metadata.fields if fld['@id'] == f['@id']), None)
        if field_obj:
            print(f"     - {field_obj['@id']} | {field_obj.get('name', '')} | {field_obj.get('dataType', '')}")
        else:
            print(f"     - {f['@id']} (field/column definition not found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rset['@id'] for rset in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records_gen = dataset.records(record_set=record_set_id)
        records = list(records_gen)
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set '{record_set_id}' with shape {df.shape}")
        else:
            print(f"Record set '{record_set_id}' returned zero records.")
    except Exception as e:
        print(f"Could not load records from '{record_set_id}':", e)

# For demo, display columns and first few rows from the first record set
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print('Fields/columns:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes.

> For all operations, **reference columns and fields by their `@id`** (not human-readable strings).

In [ ]:
import numpy as np

if len(dataframes) > 0:
    df = dataframes[main_record_set_id]

    # --- Step 1: Select a numeric field for analysis ---
    # Find a numeric field by checking the type in metadata
    record_set_obj = next((rs for rs in metadata.record_sets if rs['@id']==main_record_set_id), None)
    numeric_field_id = None
    if record_set_obj:
        all_field_ids = [f['@id'] for f in record_set_obj.get('fields', record_set_obj.get('columns', []))]
        for field_id in all_field_ids:
            field_obj = next((fld for fld in metadata.fields if fld['@id']==field_id), None)
            # Numeric types include 'schema:Float', 'schema:Integer', or similar
            if (field_obj is not None) and (field_obj.get('dataType', '') in ['schema:Float','schema:Integer','schema:Number']):
                if field_id in df.columns:
                    # Needs to exist in dataframe
                    numeric_field_id = field_id
                    print(f"Selected numeric field: '{numeric_field_id}' (name={field_obj.get('name','')}) for analysis.")
                    break
    if numeric_field_id is None:
        print("No numeric field found in first record set.")
    else:
        # Try to coerce numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # --- Step 2: Filtering records ---
        # Choose a threshold (e.g., mean value for demo)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # --- Step 3: Normalization ---
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # --- Step 4: Grouping by a categorical field (if available) ---
        group_field_id = None
        # find a categorical (Text/String) field
        for field_id in all_field_ids:
            field_obj = next((fld for fld in metadata.fields if fld['@id']==field_id), None)
            if (field_obj is not None) and (field_obj.get('dataType', '').lower() in ['schema:text', 'schema:string','schema:Text']):
                if field_id in df.columns and field_id != numeric_field_id:
                    group_field_id = field_id
                    print(f"Found grouping field '{group_field_id}' (name={field_obj.get('name', '')})")
                    break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print('No dataframes loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        # Show boxplot by group
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**

- Loaded metadata and data from the FAIR\(^2\) Croissant dataset describing protein abundance and mass spectrometry features from extracellular vesicles derived from stimulated human mast cells.
- Explored the available record sets and fields, referenced strictly by their `@id` as per best practice.
- Demonstrated numeric field filtering, normalization, and grouping.
- Visualized numeric data distributions and group comparisons.

This workflow provides a robust, reproducible way to process FAIR Croissant datasets, laying the foundation for customized analysis pipelines.